In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 切換路徑

In [15]:
cd /content/drive/MyDrive/Python與AI影像辨識應用班第02期(2024)/demo/3_object_detection

/content/drive/MyDrive/Python與AI影像辨識應用班第02期(2024)/demo/3_object_detection


In [36]:
import numpy as np
import torch
import time
import cv2
from numpy import random
from models.experimental import attempt_load
from utils.general import check_img_size, check_requirements, check_imshow, non_max_suppression, apply_classifier, \
    scale_coords, xyxy2xywh, strip_optimizer, set_logging, increment_path
from utils.plots import plot_one_box

from fastai.vision.all import *
%matplotlib inline

In [27]:
def letterbox(img, new_shape=(640, 640), color=(114, 114, 114), auto=True, scaleFill=False, scaleup=True, stride=32):
    # Resize and pad image while meeting stride-multiple constraints
    shape = img.shape[:2]  # current shape [height, width]
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)

    # Scale ratio (new / old)
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    if not scaleup:  # only scale down, do not scale up (for better test mAP)
        r = min(r, 1.0)

    # Compute padding
    ratio = r, r  # width, height ratios
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]  # wh padding
    if auto:  # minimum rectangle
        dw, dh = np.mod(dw, stride), np.mod(dh, stride)  # wh padding
    elif scaleFill:  # stretch
        dw, dh = 0.0, 0.0
        new_unpad = (new_shape[1], new_shape[0])
        ratio = new_shape[1] / shape[1], new_shape[0] / shape[0]  # width, height ratios

    dw /= 2  # divide padding into 2 sides
    dh /= 2

    if shape[::-1] != new_unpad:  # resize
        img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)  # add border
    return img, ratio, (dw, dh)

def detect_modify(img0, model, conf=0.4, imgsz=640, conf_thres = 0.25, iou_thres=0.45):
    # st.image(img0, caption="Your image", use_column_width=True)

    stride = int(model.stride.max())  # model stride
    imgsz = check_img_size(imgsz, s=stride)  # check img_size

    # Padded resize
    img0 = cv2.cvtColor(np.asarray(img0), cv2.COLOR_RGB2BGR)
    img = letterbox(img0, imgsz, stride=stride)[0]
    # Convert
    img = img[:, :, ::-1].transpose(2, 0, 1)  # BGR to RGB, to 3x416x416
    img = np.ascontiguousarray(img)


    # Get names and colors
    names = model.module.names if hasattr(model, 'module') else model.names

    #======================設定顏色======================
    colors = [[random.randint(0, 255) for _ in range(3)] for _ in names]
    #=====================================================

    # Run inference
    old_img_w = old_img_h = imgsz
    old_img_b = 1

    t0 = time.time()
    img = torch.from_numpy(img).to(device)
    # img /= 255.0  # 0 - 255 to 0.0 - 1.0
    img = img/255.0
    if img.ndimension() == 3:
        img = img.unsqueeze(0)

    # Inference
    # t1 = time_synchronized()
    with torch.no_grad():   # Calculating gradients would cause a GPU memory leak
        pred = model(img)[0]
    # t2 = time_synchronized()

    # Apply NMS
    pred = non_max_suppression(pred, conf_thres, iou_thres)
    # t3 = time_synchronized()

    # Process detections
    # for i, det in enumerate(pred):  # detections per image

    gn = torch.tensor(img0.shape)[[1, 0, 1, 0]]  # normalization gain whwh

    det = pred[0]
    if len(det):
        # Rescale boxes from img_size to im0 size
        det[:, :4] = scale_coords(img.shape[2:], det[:, :4], img0.shape).round()

        # Print results
        s = ''
        for c in det[:, -1].unique():
            n = (det[:, -1] == c).sum()  # detections per class
            s += f"{n} {names[int(c)]}{'s' * (n > 1)}, "  # add to string

        # Write results
        for *xyxy, conf, cls in reversed(det):
            label = f'{names[int(cls)]} {conf:.2f}'
            plot_one_box(xyxy, img0, label=label, color=colors[int(cls)], line_thickness=1)


    img0 = cv2.cvtColor(np.asarray(img0), cv2.COLOR_BGR2RGB)
    # st.image(img0, caption="Prediction Result", use_column_width=True)

    return img0, det

In [28]:
#set paramters
weight_path = './yolov7.pt'
imgsz = 640
conf = 0.4
conf_thres = 0.25
iou_thres=0.45
device = torch.device("cpu")

In [29]:
# Load model
model = attempt_load(weight_path, map_location=torch.device('cpu'))  # load FP32 model

Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block


In [30]:
path = '/content/drive/MyDrive/Python與AI影像辨識應用班第02期(2024)/share_dataset/pedestrian/pedestrian.v2i.yolov7pytorch/test/images/129badf1-e37f-4f40-8f35-9789b0f6bc69_jpg.rf.9d57a15d0f5355212c5718b5a2dfde18.jpg'
img = PILImage.create(path)
img0, det = detect_modify(img, model, conf=conf, imgsz=imgsz, conf_thres=conf_thres, iou_thres=iou_thres)

In [44]:
det

tensor([[1.34400e+03, 1.14400e+03, 1.62500e+03, 1.30500e+03, 9.24158e-01, 2.00000e+00],
        [6.48000e+02, 1.08900e+03, 7.42000e+02, 1.23900e+03, 8.56456e-01, 0.00000e+00],
        [8.65000e+02, 6.61000e+02, 9.80000e+02, 9.32000e+02, 8.49934e-01, 9.00000e+00],
        [6.51000e+02, 1.15800e+03, 7.36000e+02, 1.25600e+03, 8.29551e-01, 1.00000e+00],
        [2.40000e+02, 8.78000e+02, 3.01000e+02, 9.84000e+02, 8.22361e-01, 9.00000e+00],
        [7.79000e+02, 9.61000e+02, 8.30000e+02, 1.04700e+03, 6.40292e-01, 9.00000e+00],
        [9.82000e+02, 9.58000e+02, 1.01000e+03, 1.01900e+03, 6.21297e-01, 9.00000e+00],
        [1.02200e+03, 9.29000e+02, 1.06600e+03, 1.02300e+03, 5.71725e-01, 9.00000e+00],
        [1.02400e+03, 7.94000e+02, 1.05700e+03, 8.88000e+02, 5.56597e-01, 9.00000e+00],
        [1.85900e+03, 8.31000e+02, 1.92100e+03, 9.93000e+02, 5.09947e-01, 9.00000e+00],
        [1.80400e+03, 8.77000e+02, 1.85900e+03, 9.88000e+02, 4.23773e-01, 9.00000e+00]])

In [43]:
plt.imsave('test.png', img0)